# <job_name>

- 用途：
- 輸入 table：`{catalog}.{schema}.<source>`
- 輸出 table：`{catalog}.{schema}.<target>`
- 排程：
- 負責人 / 更新日期：

Cell 標籤規則見 `docs/conventions.md`。


In [ ]:
# [c01] params
# 預設值請對照 config/project.yml；job 執行時由 job parameters 覆蓋。
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("env", "dev")
dbutils.widgets.text("run_date", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
env = dbutils.widgets.get("env")
run_date = dbutils.widgets.get("run_date")
assert catalog and schema, "catalog / schema 不可為空"


In [ ]:
# [c02] imports
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


In [ ]:
# [c03] load_source
src_table = f"{catalog}.{schema}.<source>"
df_src = spark.table(src_table)
print(src_table, df_src.count())


In [ ]:
# [c04] transform_example
def transform_example(df: DataFrame) -> DataFrame:
    """範例純函式：清理 customer_id、轉日期、彙總金額。可被 tests/ 載入測試。"""
    return (
        df.withColumn("customer_id", F.upper(F.trim("customer_id")))
        .withColumn("order_date", F.to_date("order_date"))
        .withColumn("amount", F.coalesce("amount", F.lit(0)))
        .groupBy("customer_id")
        .agg(F.max("order_date").alias("order_date"), F.sum("amount").alias("amount"))
    )


In [ ]:
# [c10] run
df_out = transform_example(df_src)


In [ ]:
# [c20] write_target
# 寫入策略：overwrite（每日全量重算）。若改為增量請換 MERGE 並註明鍵值。
tgt_table = f"{catalog}.{schema}.<target>"
df_out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tgt_table)


In [ ]:
# [c30] check_target
df_chk = spark.table(tgt_table)
n_rows = df_chk.count()
n_dup = df_chk.groupBy("customer_id").count().filter("count > 1").count()
print(f"rows={n_rows} dup_keys={n_dup}")
assert n_dup == 0, "customer_id 重複"
